# Lab 5 — The evaluation harness

*Day 3, hour 3 · 50 minutes · pairs, then the whole class*

::: {.callout-note appearance="simple"}
**Objective** — stand the harness up over the golden set, calibrate the
groundedness judge against human labels, and watch the gate block a regression that
code review cannot see.

**Before you start** — Module 4's lab complete, with both corpus numbers recorded
and two misses written down.

**You finish with** — a green suite, a qualified judge (κ ≥ 0.6), a working gate,
and `EVALUATION_REPORT.md`.
:::

In [1]:
import os, pathlib, sys, re, subprocess, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + r"\[[0-9;]*m")

for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (cand / "src" / "murshid").is_dir():
        os.chdir(cand); break
    if (cand / "murshid" / "src" / "murshid").is_dir():
        os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

def run(*args, quiet_logs=True, may_fail=False):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.

    may_fail=True for the commands whose job is to exit non-zero: the gate when
    it blocks, and the uncalibrated judge. Everywhere else a non-zero exit stops
    the notebook, because a traceback printed into a page that still reports as
    executed is worse than no output at all.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        # Structured logs come in two shapes — the console format on a laptop and
        # JSON lines in the container — so drop both, rather than whichever one
        # the machine that built this notebook happened to emit.
        def _is_log(line):
            if line.startswith("20") and "[" in line[:40]:
                return True
            return line.lstrip().startswith('{"') and (
                '"stage"' in line or '"event"' in line or '"logger"' in line)
        text = "\n".join(l for l in text.splitlines() if not _is_log(l))
    else:
        # The log is the lesson here, but not all of it: assistant_built and the
        # per-call llm_cost records are plumbing, and they are also the widest
        # lines on the page. Keep the retries, the failover and the refusals.
        NOISE = ("llm_cost", "assistant_built")
        text = "\n".join(l for l in text.splitlines()
                          if not any(n in l for n in NOISE))
    print(text.strip())
    if out.returncode and not may_fail:
        # A failing subprocess does not fail the notebook on its own, so say so
        # loudly. Without this a broken command is a traceback in the middle of a
        # page that still reports as executed cleanly.
        raise SystemExit(f"command failed with exit code {out.returncode}: {' '.join(args)}")
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

# demo_v0.py is deliberately naive — hardcoded model, inline prompt, no timeout —
# but it does read OPENAI_BASE_URL, and its default is only right on a laptop.
# Point it at the same gateway as everything else so the lab works in both places.
os.environ.setdefault("OPENAI_BASE_URL", GATEWAY + "/v1")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

try:
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=3) as r:
        print("gateway:", json.load(r)["models"])
except Exception:
    print(f"gateway at {GATEWAY} is NOT answering — start it first:")
    print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


## 1 · Absorb the corpora (8 min)

The labs have been building the golden set all week without saying so. Merge them,
and read the histogram for what is **thin**, not for what is big.

In [2]:
run("eval/build_golden.py")

wrote 126 cases to eval/golden/regression_set.yaml
strata histogram:
  difficulty=hard          ################################################## 71
  difficulty=routine       ################################################## 55
  intent=escalate          ######## 8
  intent=faq               ################################################## 64
  intent=safety            ########################################## 42
  intent=service           ############ 12
  language=ar              ################################################## 64
  language=en              ################################################## 62
  risk=false_positive      ########## 10
  risk=normal              ################################################## 74
  risk=safety              ########################################## 42


0

`intent=service` has 12 cases; `intent=escalate` has 8. Adding three defensible
cases to the thinnest stratum is worth more than adding thirty to `faq`.

## 2 · Run it, and read the slices (10 min)

In [3]:
run("eval/harness.py", "--label", "default")

────────────────────────────────────────────────────────────────────────
eval | route=default | 126 cases | pass 126/126 (100%) | 13.8s | 58.8 halalas
────────────────────────────────────────────────────────────────────────
  language    ar 100% | en 100%
  intent      escalate 100% | faq 100% | safety 100% | service 100%
  difficulty  hard 100% | routine 100%
  risk        false_positive 100% | normal 100% | safety 100%

  written: /srv/eval/out/eval_default.json


0

In [4]:
run("eval/harness.py", "--label", "vllm", "--route", "vllm")

────────────────────────────────────────────────────────────────────────
eval | route=vllm | 126 cases | pass 118/126 (94%) | 24.5s | 26.0 halalas
────────────────────────────────────────────────────────────────────────
  language    ar 92% | en 95%
  intent      escalate 100% | faq 92% | safety 100% | service 75%
  difficulty  hard 94% | routine 93%
  risk        false_positive 100% | normal 89% | safety 100%

  8 failing:
    g037 [normal] ar in-directory — appointment_booking — intent(intent was 'service'), contains(missing 'بدون رسوم')
    g042 [normal] en out-of-directory — must not guess a fee — regex(no match for "(don't have|do not have) that information"), pytho
    g047 [normal] en out-of-directory — must not guess a fee — regex(no match for "(don't have|do not have) that information"), pytho
    g051 [normal] ar out-of-directory — must not guess a fee — regex(no match for 'لا تتوفر لدي هذه المعلومة'), python(amounts not in
    g054 [normal] ar out-of-directory — must not gue

0

Find the stratum where the open-weight route loses most. It is `service` — tool
contracts, not prose — and that single observation is what shapes Module 6's routing
table. The average would have told you nothing useful.

Notice also that **safety is 100% on both**. The safety stratum tests the guards,
which are application code. A weaker model does not weaken them.

## 3 · Calibrate the judge (12 min) — the centrepiece

Label first, in public: your cohort labels 40 groundedness cases between you, then
argues the pool out loud. **Settle the human disagreements first**, before anyone
looks at a judge.

In [5]:
run("eval/build_human_labels.py")

wrote 40 labelled answers to eval/golden/human_labels_40.jsonl
  label distribution: 0.0: 12, 0.5: 6, 1.0: 22


0

The seeded rubric is deliberately vague, and the run says so.

In [6]:
run("eval/calibrate_judge.py", "--rubric", "groundedness.v1.md", may_fail=True)

rubric: groundedness.v1.md
  agreement: 62% | cohen_kappa: 0.35 over 40 cases
  VERDICT: rubric needs work — do NOT wire this judge to anything

  15 disagreements — read them, then fix the RUBRIC:
    h002: human 1.0 vs judge 0.5 — The answer reads well.
    h004: human 1.0 vs judge 0.5 — The answer reads well.
    h005: human 1.0 vs judge 0.5 — The answer reads well.
    h017: human 1.0 vs judge 0.5 — No fee claim to check; claims are directory-shaped but imprecise.
    h018: human 1.0 vs judge 0.5 — No fee claim to check; claims are directory-shaped but imprecise.
    h023: human 0.0 vs judge 1.0 — The answer reads well.


1

Read the evidence column. *"The answer reads well"* is not evidence; it is a mood.
The sharpened rubric adds anchors, a required evidence quote, and a clause for the
answer that correctly declines to answer.

In [7]:
run("eval/calibrate_judge.py", "--rubric", "groundedness.v2.md")

rubric: groundedness.v2.md
  agreement: 90% | cohen_kappa: 0.84 over 40 cases
  VERDICT: judge may gate (tracking)

  4 disagreements — read them, then fix the RUBRIC:
    h017: human 1.0 vs judge 0.5 — No fee claim to check; claims are directory-shaped but imprecise.
    h018: human 1.0 vs judge 0.5 — No fee claim to check; claims are directory-shaped but imprecise.
    h019: human 1.0 vs judge 0.5 — No fee claim to check; claims are directory-shaped but imprecise.
    h020: human 1.0 vs judge 0.5 — No fee claim to check; claims are directory-shaped but imprecise.


0

::: {.callout-important}
## Fix the rubric, not the humans

The difference between those two runs is **a rubric edit and nothing else**. A judge
is an instrument you qualify, not an oracle you consult.

Four disagreements survive, all on answers with no fee to check: the rubric still
does not say what to do when there is no number. That is what a v3 fixes, and
noticing it is the exercise.
:::

Here is why κ and not percent agreement, in twelve lines:

In [8]:
from eval.calibrate_judge import cohen_kappa

human = ["1.0"] * 36 + ["0.0"] * 4
lazy  = ["1.0"] * 40            # an instrument that always says 1.0
print("percent agreement:", sum(a == b for a, b in zip(human, lazy)) / len(human))
print("cohen kappa      :", cohen_kappa(human, lazy))

percent agreement: 0.9
cohen kappa      : 0.0


Ninety percent agreement, and it knows nothing. That subtraction is the whole
argument for κ.

## 4 · Wire the gate, then break it (8 min)

The baseline is already promoted in this checkout, so the gate has something to
compare against. A green run first:

In [9]:
run("eval/gate.py", "eval/out/eval_default.json", "--baseline", "eval/baseline.json")

| stratum | baseline | this run | delta |
|---|---|---|---|
| **overall** | 100% | 100% | +0.0pt |
| language=ar | 100% | 100% | +0.0pt |
| language=en | 100% | 100% | +0.0pt |
| intent=escalate | 100% | 100% | +0.0pt |
| intent=faq | 100% | 100% | +0.0pt |
| intent=safety | 100% | 100% | +0.0pt |
| intent=service | 100% | 100% | +0.0pt |
| difficulty=hard | 100% | 100% | +0.0pt |
| difficulty=routine | 100% | 100% | +0.0pt |
| risk=false_positive | 100% | 100% | +0.0pt |
| risk=normal | 100% | 100% | +0.0pt |
| risk=safety | 100% | 100% | +0.0pt |

PASS: overall +0.0pt | worst stratum difficulty=hard +0.0pt | safety 100%


0

Now the seeded prompt change. `answer_faq.v6` is friendlier. It reads better. It
also quietly drops the don't-know rule.

In [10]:
import os
os.environ["MURSHID_FAQ_PROMPT"] = "answer_faq.v6"
run("eval/harness.py", "--label", "seeded")
del os.environ["MURSHID_FAQ_PROMPT"]

────────────────────────────────────────────────────────────────────────
eval | route=default | 126 cases | pass 118/126 (94%) | 13.9s | 61.1 halalas
────────────────────────────────────────────────────────────────────────
  language    ar 94% | en 94%
  intent      escalate 100% | faq 88% | safety 100% | service 100%
  difficulty  hard 89% | routine 100%
  risk        false_positive 100% | normal 89% | safety 100%

  8 failing:
    g043 [normal] en out-of-directory — must not guess a fee — regex(no match for "(don't have|do not have) that information"), pytho
    g045 [normal] en out-of-directory — must not guess a fee — regex(no match for "(don't have|do not have) that information"), pytho
    g046 [normal] en out-of-directory — must not guess a fee — regex(no match for "(don't have|do not have) that information"), pytho
    g047 [normal] en out-of-directory — must not guess a fee — regex(no match for "(don't have|do not have) that information"), pytho
    g048 [normal] ar out-of-dir

In [11]:
run("eval/gate.py", "eval/out/eval_seeded.json", "--baseline", "eval/baseline.json", may_fail=True)

| stratum | baseline | this run | delta |
|---|---|---|---|
| **overall** | 100% | 94% | -6.3pt |
| language=ar | 100% | 94% | -6.2pt |
| language=en | 100% | 94% | -6.5pt |
| intent=escalate | 100% | 100% | +0.0pt |
| intent=faq | 100% | 88% | -12.5pt |
| intent=safety | 100% | 100% | +0.0pt |
| intent=service | 100% | 100% | +0.0pt |
| difficulty=hard | 100% | 89% | -11.3pt |
| difficulty=routine | 100% | 100% | +0.0pt |
| risk=false_positive | 100% | 100% | +0.0pt |
| risk=normal | 100% | 89% | -10.8pt |
| risk=safety | 100% | 100% | +0.0pt |

BLOCKED:
  overall: overall 94% vs baseline 100% (-6.3pt, margin 2pt)
  slice:language=ar: language=ar 94% vs baseline 100% (-6.2pt, margin 3pt)
  slice:language=en: language=en 94% vs baseline 100% (-6.5pt, margin 3pt)
  slice:intent=faq: intent=faq 88% vs baseline 100% (-12.5pt, margin 3pt)
  slice:difficulty=hard: difficulty=hard 89% vs baseline 100% (-11.3pt, margin 3pt)
  slice:risk=normal: risk=normal 89% vs baseline 100% (-10.8pt, margi

1

Read the slice rows before the overall row, and sit with what they say.

**Code review sees a tone change. The gate sees eight invented fees.**

## 5 · The report (4 min)

In [12]:
run("eval/report.py")

wrote EVALUATION_REPORT.md (70 lines)


0

Read `EVALUATION_REPORT.md` once, now, because it is the capstone's headline
deliverable in miniature. Look especially at the last section — **known
limitations** — and understand that it is there because honesty scores points, and
because a report with no limitations section is a report nobody believes.